# F-18 HARV — Aggressive Trajectory Generation
## OCP Transition Library + SAC Reinforcement Learning
### All 275 trim states loaded from `.mat` | OCP-as-reward | No behaviour cloning | No adversary

**Changes from original notebook:**
- Trim library: 275 states directly from `trim_states.mat` / `trim_controls.mat`
- Transition library: loaded from `transition_library.mat` (31,125 pre-solved OCP transitions)
- Objective: pure aggressiveness (`-w_agg · Σ(p²+q²+r²)·dt`) — no adversary
- Reward: OCP-as-reward using **(s_curr, s_next, s_goal)** triplet — no behaviour cloning
- Observation: 39-D `[x_curr | x_goal | Δx]`
- Inference: ~0.04 s for any (x0, xf) pair


In [1]:
import numpy as np
import scipy.io as sio
import scipy.integrate as sci
from scipy.interpolate import interp1d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random, os, time, copy, warnings
warnings.filterwarnings('ignore')

# optional CasADi import (only needed if you want to run new OCP solves)
try:
    import casadi as ca
    HAS_CASADI = True
    print(f"  casadi {ca.__version__}")
except ImportError:
    HAS_CASADI = False
    print("  CasADi not found — OCP re-solve disabled (transition_library.mat used directly)")

SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Imports OK  |  device={DEVICE}  |  torch={torch.__version__}  |  numpy={np.__version__}")


  CasADi not found — OCP re-solve disabled (transition_library.mat used directly)
✓ Imports OK  |  device=cpu  |  torch=2.11.0+cpu  |  numpy=2.0.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ═══════════════════════════════════════════════════════════════════════
#  F-18 HARV Physical & Aerodynamic Parameters
#  Source: aircraft struct in transition_library.mat (verified match)
# ═══════════════════════════════════════════════════════════════════════
class F18Params:
    mass = 15096.5; Ixx = 31183.6; Iyy = 205127.5; Izz = 230432.1; Ixz = 4028.0
    S_ref = 37.16; c_bar = 3.511; b = 11.404
    CL_alpha = 5.0; CD0 = 0.020; k = 0.080; k_alpha2 = 0.300
    Cm_alpha = -0.45; Cm_q = -4.50; Cm_delta_e = -0.50
    Cl_beta = -0.080; Cl_p = -0.300; Cl_r = 0.050; Cl_delta_a = -0.150; Cl_delta_r = 0.050
    Cn_beta = 0.080; Cn_p = -0.030; Cn_r = -0.150; Cn_delta_a = -0.010; Cn_delta_r = -0.120
    CY_beta = -0.730; CY_dr = 0.200
    delta_e_min = np.deg2rad(-24.0);  delta_e_max = np.deg2rad(10.5)
    delta_a_min = np.deg2rad(-25.0);  delta_a_max = np.deg2rad(45.0)
    delta_r_min = np.deg2rad(-30.0);  delta_r_max = np.deg2rad(30.0)
    thrust_min  = 500.0;              thrust_max  = 143200.0

A = F18Params()
g_acc = 9.81;  rho = 1.225
print("✓ F-18 HARV parameters loaded")
print(f"  mass={A.mass} kg  |  S={A.S_ref} m²  |  b={A.b} m  |  c̄={A.c_bar} m")
print(f"  T ∈ [{A.thrust_min:.0f}, {A.thrust_max:.0f}] N")


✓ F-18 HARV parameters loaded
  mass=15096.5 kg  |  S=37.16 m²  |  b=11.404 m  |  c̄=3.511 m
  T ∈ [500, 143200] N


In [4]:
# ═══════════════════════════════════════════════════════════════════════
#  6-DOF EOM — NumPy (for Gym env / RK4 step)
#  State: [u,v,w, p,q,r, q0,q1,q2,q3, xN,yE,zD]   (13-D, quat scalar-first)
#  Ctrl:  [T, δe, δa, δr]                            (4-D)
# ═══════════════════════════════════════════════════════════════════════

def quat_to_dcm(q):
    """Body→Inertial DCM from quaternion [q0,q1,q2,q3] (scalar-first)."""
    q0,q1,q2,q3 = q
    return np.array([
        [1-2*(q2**2+q3**2),   2*(q1*q2-q0*q3),   2*(q1*q3+q0*q2)],
        [  2*(q1*q2+q0*q3), 1-2*(q1**2+q3**2),   2*(q2*q3-q0*q1)],
        [  2*(q1*q3-q0*q2),   2*(q2*q3+q0*q1), 1-2*(q1**2+q2**2)]
    ])

def f18_6dof_numpy(x, ctrl):
    u_b,v_b,w_b = x[0],x[1],x[2]
    p,  q,  r   = x[3],x[4],x[5]
    q0,q1,q2,q3 = x[6],x[7],x[8],x[9]
    T, de, da, dr = np.clip(ctrl[0],A.thrust_min,A.thrust_max),                     np.clip(ctrl[1],A.delta_e_min,A.delta_e_max),                     np.clip(ctrl[2],A.delta_a_min,A.delta_a_max),                     np.clip(ctrl[3],A.delta_r_min,A.delta_r_max)
    V    = np.sqrt(u_b**2+v_b**2+w_b**2+1e-6)
    qbar = 0.5*rho*V**2
    alp  = np.arctan2(w_b, u_b)
    bet  = np.arcsin(np.clip(v_b/V,-1,1))
    CL   = A.CL_alpha*alp
    CD   = A.CD0 + A.k*CL**2 + A.k_alpha2*alp**2
    CY   = A.CY_beta*bet + A.CY_dr*dr
    L_f  = qbar*A.S_ref*CL; D_f = qbar*A.S_ref*CD; Y_f = qbar*A.S_ref*CY
    ca_,sa_ = np.cos(alp),np.sin(alp); cb_,sb_ = np.cos(bet),np.sin(bet)
    Cwb = np.array([[ca_*cb_,sb_,sa_*cb_],[-ca_*sb_,cb_,-sa_*sb_],[-sa_,0.,ca_]])
    Fb  = Cwb.T @ np.array([T-D_f, Y_f, -L_f])
    Rbi = quat_to_dcm([q0,q1,q2,q3])
    Fg  = A.mass*(Rbi.T @ np.array([0.,0.,g_acc]))
    FX,FY_b,FZ = Fb[0]+Fg[0], Fb[1]+Fg[1], Fb[2]+Fg[2]
    u_dot = FX/A.mass + r*v_b - q*w_b
    v_dot = FY_b/A.mass + p*w_b - r*u_b
    w_dot = FZ/A.mass + q*u_b - p*v_b
    bV,cV = A.b/(2*V), A.c_bar/(2*V)
    Cl = A.Cl_beta*bet+A.Cl_p*bV*p+A.Cl_r*bV*r+A.Cl_delta_a*da+A.Cl_delta_r*dr
    Cm = A.Cm_alpha*alp+A.Cm_q*cV*q+A.Cm_delta_e*de
    Cn = A.Cn_beta*bet+A.Cn_p*bV*p+A.Cn_r*bV*r+A.Cn_delta_a*da+A.Cn_delta_r*dr
    Lm = qbar*A.S_ref*A.b*Cl; Mm = qbar*A.S_ref*A.c_bar*Cm; Nm = qbar*A.S_ref*A.b*Cn
    I3 = np.array([[A.Ixx,0,-A.Ixz],[0,A.Iyy,0],[-A.Ixz,0,A.Izz]])
    om = np.array([p,q,r])
    om_dot = np.linalg.solve(I3, np.array([Lm,Mm,Nm])-np.cross(om,I3@om))
    Om = 0.5*np.array([[0,-p,-q,-r],[p,0,r,-q],[q,-r,0,p],[r,q,-p,0]])
    qd = Om @ np.array([q0,q1,q2,q3])
    vi = Rbi @ np.array([u_b,v_b,w_b])
    return np.array([u_dot,v_dot,w_dot,om_dot[0],om_dot[1],om_dot[2],
                     qd[0],qd[1],qd[2],qd[3],vi[0],vi[1],vi[2]])

def rk4_step(x, u, dt):
    k1 = f18_6dof_numpy(x,u)
    k2 = f18_6dof_numpy(x+dt/2*k1,u)
    k3 = f18_6dof_numpy(x+dt/2*k2,u)
    k4 = f18_6dof_numpy(x+dt*k3,u)
    xn = x + (dt/6)*(k1+2*k2+2*k3+k4)
    xn[6:10] /= np.linalg.norm(xn[6:10])+1e-12   # keep quaternion unit-norm
    return xn

print("✓ F-18 6-DOF EOM (NumPy + RK4 stepper) defined")


✓ F-18 6-DOF EOM (NumPy + RK4 stepper) defined


In [7]:
# ═══════════════════════════════════════════════════════════════════════
#  Load 275 trim states + controls + pre-solved transition library
#  from the MATLAB .mat files
# ═══════════════════════════════════════════════════════════════════════

import scipy.io as sio
import numpy as np

def load_maneuver_library(mat_path='maneuver_library.mat'):
    """
    Returns:
        list of dictionaries

    Each entry contains:
        {
            'maneuver_type'      : str,
            'trajectory_time'    : (N,) array,
            'trajectory_states'  : (13,N) array,
            'trajectory_controls': (4,N) array,
            'duration'           : float,
            'trim_state'         : (13,) array,
            'trim_controls'      : (4,) array,
            'heading_deg'        : float,
            'idx'                : int
        }
    """

    mat = sio.loadmat(mat_path, squeeze_me=False)

    raw_lib = mat['maneuver_library']

    maneuver_lib = []

    for i in range(raw_lib.shape[1]):

        m = raw_lib[0, i][0, 0]

        maneuver_type = str(m['maneuver_type'][0])

        trajectory_time = np.asarray(
            m['trajectory_time']
        ).flatten().astype(np.float64)

        trajectory_states = np.asarray(
            m['trajectory_states']
        ).astype(np.float64)

        trajectory_controls = np.asarray(
            m['trajectory_controls']
        ).astype(np.float64)

        duration = float(
            np.asarray(m['duration']).squeeze()
        )

        trim_state = np.asarray(
            m['trim_state']
        ).flatten().astype(np.float64)

        ctrl = m['trim_controls'][0, 0]

        trim_controls = np.array([
            float(np.asarray(ctrl['thrust']).squeeze()),
            float(np.asarray(ctrl['delta_e']).squeeze()),
            float(np.asarray(ctrl['delta_a']).squeeze()),
            float(np.asarray(ctrl['delta_r']).squeeze())
        ])

        heading_deg = float(
            np.asarray(m['heading_deg']).squeeze()
        )

        maneuver_lib.append({
            'maneuver_type': maneuver_type,
            'trajectory_time': trajectory_time,
            'trajectory_states': trajectory_states,
            'trajectory_controls': trajectory_controls,
            'duration': duration,
            'trim_state': trim_state,
            'trim_controls': trim_controls,
            'heading_deg': heading_deg,
            'idx': i
        })

    return maneuver_lib


def load_transition_library(mat_path='transition_library.mat'):
    """
    Parses the (275×275) structured array from the .mat file.
    Returns dict keyed by (i,j) → {'feasible','X','U','T','Tf','cost','status'}
    Also returns list of feasible (i,j) tuples.
    """
    ml = sio.loadmat(mat_path, squeeze_me=False)
    tl_raw = ml['transition_library']   # (275,275) structured numpy array

    trans = {}
    feasible_pairs = []

    for i in range(25):
        for j in range(25):
            entry = tl_raw[i, j]
            feas = int(entry['feasible'].flat[0])
            if feas == 1:
                X  = entry['X'].squeeze()      # (81,13) — may need reshape
                U  = entry['U'].squeeze()      # (81,4)
                T  = entry['T'].flatten()      # (81,)
                Tf = float(entry['Tf'].flat[0])
                cost = float(entry['cost'].flat[0])
                st = str(entry['solver_status'].flat[0])
                # Ensure correct shapes
                if X.ndim == 2 and X.shape == (81,13):
                    pass
                elif X.ndim == 2 and X.shape == (13,81):
                    X = X.T
                if U.ndim == 2 and U.shape == (81,4):
                    pass
                elif U.ndim == 2 and U.shape == (4,81):
                    U = U.T
                trans[(i,j)] = dict(feasible=True, X=X, U=U, T=T,
                                    Tf=Tf, cost=cost, status=st)
                feasible_pairs.append((i,j))
            else:
                trans[(i,j)] = dict(feasible=False, X=None, U=None, T=None,
                                    Tf=np.nan, cost=np.nan, status='not_attempted')
    return trans, feasible_pairs


# --- Load everything ---
print("Loading trim states & controls ...")
TRIM_LIBRARY = load_maneuver_library('/content/drive/My Drive/maneuver_library.mat'
    )
N_TRIM = len(TRIM_LIBRARY)

print("Loading transition library  — this takes a few seconds ...")
TRANSITION_LIBRARY, FEASIBLE_PAIRS = load_transition_library(
    '/content/drive/My Drive/transition_library.mat')

print(f"\n✓ Trim library   : {N_TRIM} states")
print(f"✓ Transition lib  : {len(FEASIBLE_PAIRS):,} feasible pairs  "
      f"({100*len(FEASIBLE_PAIRS)/(275*274):.1f}% of {275*274:,} possible)")

# Quick sanity check
s0 = TRIM_LIBRARY[0]['trim_state']
print(f"\nSample trim state [0]: u={s0[0]:.2f} m/s  w={s0[2]:.2f} m/s  "
      f"q0={s0[6]:.4f}  alt={-s0[12]:.0f} m")

ex = TRANSITION_LIBRARY[(0,1)]
if ex['feasible']:
    print(f"Sample OCP (0→1)   : Tf={ex['Tf']:.2f}s  cost={ex['cost']:.4f}  "
          f"X.shape={ex['X'].shape}  U.shape={ex['U'].shape}")


Loading trim states & controls ...
Loading transition library  — this takes a few seconds ...

✓ Trim library   : 25 states
✓ Transition lib  : 450 feasible pairs  (0.6% of 75,350 possible)

Sample trim state [0]: u=119.51 m/s  w=10.83 m/s  q0=0.9990  alt=4000 m
Sample OCP (0→1)   : Tf=10.00s  cost=0.3226  X.shape=(81, 13)  U.shape=(81, 4)


In [10]:
# ═══════════════════════════════════════════════════════════════════════
#  F-18 Aggressive Manoeuvre Environment
#
#  Observation (39-D, NO adversary slot):
#      [x_curr(13) | x_goal(13) | Δx(13)]   normalised by X_SCALE
#
#  Action (4-D):  normalised ∈ [-1,1] → [T, δe, δa, δr]
#
#  Reward (5-component):
#      r_prog  : ±1 dense progress toward goal
#      r_oob   : altitude floor penalty
#      r_goal  : FNPG-NH sparse terminals (+1000 / +10000)
#      r_agg   : tanh-scaled body-rate aggressiveness (PRIMARY)
#      r_ocp   : OCP imitation using (s_curr, s_next, s_goal) triplet
#      r_ctrl  : control smoothness penalty
#
#  OCP seeding strategy: 70% episodes seed from pre-solved library,
#                        30% random pairs → generalisation
# ═══════════════════════════════════════════════════════════════════════

class F18AggressiveEnv(gym.Env):
    metadata = {"render_modes": []}

    # State normalisation scales
    X_SCALE = np.array([250., 50., 100.,
                        np.deg2rad(180.), np.deg2rad(60.), np.deg2rad(90.),
                        1., 1., 1., 1.,
                        5000., 5000., 5000.], dtype=np.float32)

    # Control midpoints and half-ranges for tanh normalisation
    U_MID  = np.array([(A.thrust_max+A.thrust_min)/2,
                       (A.delta_e_max+A.delta_e_min)/2,
                       (A.delta_a_max+A.delta_a_min)/2,
                       (A.delta_r_max+A.delta_r_min)/2], dtype=np.float32)
    U_HALF = np.array([(A.thrust_max-A.thrust_min)/2,
                       (A.delta_e_max-A.delta_e_min)/2,
                       (A.delta_a_max-A.delta_a_min)/2,
                       (A.delta_r_max-A.delta_r_min)/2], dtype=np.float32)

    # FNPG-NH sparse reward constants
    R_GOAL   = +1000.0   # reached goal velocity window
    R_TARGET = +10000.0  # reached tight goal tolerance
    R_OOB    = -1.0      # per-step altitude floor penalty

    def __init__(self, trim_lib, trans_lib, feasible_pairs,
                 dt=0.05, max_t=50.0,
                 ocp_seed_prob=0.70,
                 w_agg=5.0, w_ocp=2.0, w_ctrl=0.05):
        super().__init__()
        self.trim_lib  = trim_lib
        self.trans_lib = trans_lib
        self.feas      = feasible_pairs          # list of (i,j) feasible
        self.all_pairs = [(i,j) for i in range(len(trim_lib))
                                  for j in range(len(trim_lib)) if i!=j]
        self.dt          = dt
        self.max_steps   = int(max_t / dt)
        self.ocp_seed_p  = ocp_seed_prob
        self.w_agg       = w_agg
        self.w_ocp       = w_ocp
        self.w_ctrl      = w_ctrl

        self.observation_space = spaces.Box(-6., 6., shape=(39,), dtype=np.float32)
        self.action_space      = spaces.Box(-1., 1., shape=(4,),  dtype=np.float32)

        self._reset_internals()

    def _pair_difficulty(i, j, trim_lib):
      xi = trim_lib[i]['trim_state']
      xj = trim_lib[j]['trim_state']
      return np.linalg.norm(xi[10:13] - xj[10:13])   # NED distance

# Pre-sort at startup


    # ── internals ──────────────────────────────────────────────────────
    def _reset_internals(self):
        self.x = np.zeros(13); self.x_goal = np.zeros(13)
        self.u_prev = np.zeros(4); self.step_n = 0
        self._ocp = None        # current OCP reference (or None)
        self._prev_dist = np.inf
        self.i_src = 0; self.j_tgt = 0

    def _norm_x(self, x):
        return (x / self.X_SCALE).astype(np.float32)

    def _get_obs(self):
        return np.concatenate([
            self._norm_x(self.x),
            self._norm_x(self.x_goal),
            self._norm_x(self.x - self.x_goal)
        ], dtype=np.float32)                       # 39-D

    def _action_to_ctrl(self, a):
        u = self.U_MID + np.clip(a,-1,1)*self.U_HALF
        return np.clip(u,
            [A.thrust_min, A.delta_e_min, A.delta_a_min, A.delta_r_min],
            [A.thrust_max, A.delta_e_max, A.delta_a_max, A.delta_r_max])

    def _denorm_action(self, a):
        return self._action_to_ctrl(a)

    # ── OCP imitation reward using (s, s', s_goal) triplet ─────────────
    def _r_ocp_triplet(self, x_curr, x_next):
        """
        OCP-as-reward: reward = improvement in alignment with OCP reference.
        Uses s_curr, s_next, and s_goal implicitly through the OCP trajectory.
        Positive when x_next is closer to the OCP reference than x_curr.
        """
        if self._ocp is None:
            return 0.0
        t_now = self.step_n * self.dt
        T_ocp = self._ocp['T']
        idx   = int(np.searchsorted(T_ocp, t_now))
        idx   = min(idx, len(T_ocp)-1)
        x_ref = self._ocp['X'][idx]               # OCP reference at current time

        # Distance of current and next state from OCP reference (velocity+attitude subspace)
        sc = self.X_SCALE[:10]
        d_curr = np.linalg.norm((x_curr[:10] - x_ref[:10]) / sc)
        d_next = np.linalg.norm((x_next[:10] - x_ref[:10]) / sc)

        # Also consider goal distance (s_goal implicitly encoded in x_goal)
        # A step that moves both toward OCP ref AND toward goal gets a bonus
        d_goal_curr = np.linalg.norm((x_curr[:6] - self.x_goal[:6]) / self.X_SCALE[:6])
        d_goal_next = np.linalg.norm((x_next[:6] - self.x_goal[:6]) / self.X_SCALE[:6])
        goal_prog = np.clip(d_goal_curr - d_goal_next, -1, 1)  # +ve = moving toward goal

        # OCP alignment reward: improvement + goal-conditioned bonus
        r_align = self.w_ocp * (np.exp(-3.*d_next) - np.exp(-3.*d_curr))
        r_goal_cond = 0.5 * self.w_ocp * goal_prog * np.exp(-2.*d_next)
        return float(r_align + r_goal_cond)

    # ── reset ──────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_internals()

       # 70% OCP-seeded, 30% random for generalisation
        if len(self.feas) > 0 and np.random.rand() < self.ocp_seed_p:
            (i, j) = random.choice(self.feas)
            self._ocp = self.trans_lib[(i,j)]
        else:
            i, j = random.choice(self.all_pairs)
            self._ocp = None



        self.i_src = i; self.j_tgt = j
        self.x      = self.trim_lib[i]['trim_state'].copy()
        self.x_goal = self.trim_lib[j]['trim_state'].copy()
        self.u_prev = self.trim_lib[i]['trim_controls'].copy()

        # FNPG-NH reset: zero body rates, exact quaternion unit-norm
        self.x[3:6]  = 0.0
        n = np.linalg.norm(self.x[6:10]); self.x[6:10] /= max(n, 1e-12)

        self._prev_dist = np.linalg.norm(self.x[10:13] - self.x_goal[10:13])
        return self._get_obs(), {}

    # ── step ───────────────────────────────────────────────────────────
    def step(self, action):
        u     = self._action_to_ctrl(action)
        x_new = rk4_step(self.x, u, self.dt)
        self.step_n += 1

        p, q_r, r = x_new[3], x_new[4], x_new[5]

        # 1. Dense progress reward (FNPG-NH §III-B)
        dist = np.linalg.norm(x_new[10:13] - self.x_goal[10:13])
        # Define potential function
        def _potential(x, x_goal):
    # Normalise by maximum expected distance (5000 m NED)
          pos_err = np.linalg.norm(x[10:13] - x_goal[10:13]) / 5000.0
          vel_err = np.linalg.norm(x[:6]    - x_goal[:6])    / 250.0
          return -(pos_err + 0.5 * vel_err)   # higher = closer to goal

# Inside step(), replace r_prog
        gamma = 0.99
        r_shape = gamma * _potential(x_new, self.x_goal) \
                - _potential(self.x, self.x_goal)
# r_shape is positive when moving toward goal, negative when moving away
# No conflict with r_agg — both can be maximised simultaneously
        self._prev_dist = dist

        # 2. Altitude floor
        alt   = -x_new[12]
        r_oob = self.R_OOB if alt < 50.0 else 0.0

        # 3. Sparse goal terminals (FNPG-NH)
        vel_err = np.linalg.norm(x_new[:6] - self.x_goal[:6])
        in_tight = (vel_err < 3.0  and dist < 20.0)
        in_loose = (vel_err < 8.0  and dist < 60.0)
        r_goal   = (self.R_TARGET if in_tight
                    else self.R_GOAL if in_loose
                    else 0.0)

        # 4. Aggressiveness — PRIMARY objective (tanh-scaled, always has gradient)
        pqr_sq = p**2 + q_r**2 + r**2
        r_agg  = self.w_agg * float(np.tanh(pqr_sq / np.deg2rad(45)**2))

        # 5. OCP imitation using (s_curr, s_next, s_goal) triplet — NO behaviour cloning
        r_ocp  = self._r_ocp_triplet(self.x, x_new)
        phase = self.step_n / self.max_steps
        if phase > 0.9:
            r_approach = 5.0 * np.exp(-vel_err / 3.0) * np.exp(-dist / 20.0)
        else:
            r_approach = 0.0

        # 6. Control smoothness
        du     = (u - self.u_prev) / (2.0 * self.U_HALF + 1e-9)
        r_ctrl = -self.w_ctrl * float(np.sum(du**2))

        reward = float(r_shape + r_oob + r_goal + r_agg + r_ocp + r_ctrl + r_approach)

        self.x      = x_new
        self.u_prev = u

        # Inside env.step() — replace the old r_goal block

        vel_err = np.linalg.norm(x_new[:6] - self.x_goal[:6])
        dist    = np.linalg.norm(x_new[10:13] - self.x_goal[10:13])

        in_precision = (vel_err < 2.0 and dist < 10.0)
        in_tight     = (vel_err < 4.0 and dist < 30.0)
        in_loose     = (vel_err < 8.0 and dist < 60.0)

        if in_precision:
          r_goal = +15_000.0
          terminated = True
          success_score = 1.0
        elif in_tight:
          r_goal = +10_000.0
          terminated = True
          success_score = 0.8
        elif in_loose:
          r_goal = +1_000.0
          terminated = False   # keep running — still time to tighten
          success_score = 0.4
        else:
          r_goal = 0.0
          terminated = False
          success_score = 0.0

# store for info dict

        truncated  = self.step_n >= self.max_steps

        info = dict(step=self.step_n, reward=reward,
                    r_shape=r_shape, r_goal=r_goal,
                    r_agg=float(r_agg), r_ocp=float(r_ocp),
                    dist_goal=dist, vel_err=vel_err,
                    pqr_rms_deg=float(np.degrees(np.sqrt(pqr_sq/3))),
                    goal_reached=in_tight,
                    i_src=self.i_src, j_tgt=self.j_tgt)
        info['success_score'] = success_score
        info['in_precision']  = in_precision
        info['in_tight']      = in_tight
        info['in_loose']      = in_loose
        return self._get_obs(), reward, terminated, truncated, info


# Quick test
_env_test = F18AggressiveEnv(TRIM_LIBRARY, TRANSITION_LIBRARY, FEASIBLE_PAIRS)
_obs, _ = _env_test.reset()
print(f"✓ Environment OK")
print(f"  obs_space={_env_test.observation_space.shape}  (39-D: curr|goal|Δx)")
print(f"  act_space={_env_test.action_space.shape}  (4-D: T|δe|δa|δr)")
print(f"  max_steps={_env_test.max_steps}  dt={_env_test.dt}s")
_a = _env_test.action_space.sample()
_o2, _r, _t, _tr, _inf = _env_test.step(_a)
print(f"  Random step: reward={_r:.3f}  r_agg={_inf['r_agg']:.3f}  r_ocp={_inf['r_ocp']:.3f}")
del _env_test


✓ Environment OK
  obs_space=(39,)  (39-D: curr|goal|Δx)
  act_space=(4,)  (4-D: T|δe|δa|δr)
  max_steps=1000  dt=0.05s
  Random step: reward=0.572  r_agg=1.385  r_ocp=-0.784


In [11]:
# ═══════════════════════════════════════════════════════════════════════
#  SAC Neural Networks  (FNPG-NH §IV: 400×300 hidden layers)
# ═══════════════════════════════════════════════════════════════════════

OBS_DIM = 39    # 39-D — no adversary slot
ACT_DIM = 4
HIDDEN  = (400, 300)

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=(400,300), act=nn.ReLU):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), act()]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


class GaussianActor(nn.Module):
    LOG_STD_MIN, LOG_STD_MAX = -5, 2
    def __init__(self, obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden=HIDDEN):
        super().__init__()
        self.base = MLP(obs_dim, hidden[-1], hidden[:-1])
        self.mu_head    = nn.Linear(hidden[-1], act_dim)
        self.log_std_head = nn.Linear(hidden[-1], act_dim)

    def forward(self, obs, deterministic=False, with_log_prob=True):
        h   = self.base(obs)
        mu  = self.mu_head(h)
        if deterministic:
            a_tanh = torch.tanh(mu)
            return a_tanh, None
        log_std = self.log_std_head(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        u    = dist.rsample()
        a_tanh = torch.tanh(u)
        if with_log_prob:
            logp = dist.log_prob(u) - torch.log(1 - a_tanh**2 + 1e-6)
            logp = logp.sum(-1, keepdim=True)
        else:
            logp = None
        return a_tanh, logp


class TwinCritic(nn.Module):
    def __init__(self, obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden=HIDDEN):
        super().__init__()
        self.q1 = MLP(obs_dim+act_dim, 1, hidden)
        self.q2 = MLP(obs_dim+act_dim, 1, hidden)

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)

    def q_min(self, obs, act):
        q1,q2 = self.forward(obs, act)
        return torch.min(q1, q2)


class ReplayBuffer:
    def __init__(self, capacity=1_000_000, obs_dim=OBS_DIM, act_dim=ACT_DIM):
        self.cap = capacity; self.ptr = 0; self.size = 0
        self.obs  = np.zeros((capacity, obs_dim),  dtype=np.float32)
        self.act  = np.zeros((capacity, act_dim),  dtype=np.float32)
        self.rew  = np.zeros((capacity, 1),         dtype=np.float32)
        self.obs2 = np.zeros((capacity, obs_dim),  dtype=np.float32)
        self.done = np.zeros((capacity, 1),         dtype=np.float32)

    def push(self, o, a, r, o2, d):
        k = self.ptr
        self.obs[k]=o; self.act[k]=a; self.rew[k]=r; self.obs2[k]=o2; self.done[k]=d
        self.ptr = (k+1) % self.cap
        self.size = min(self.size+1, self.cap)

    def sample(self, batch=256):
        idx = np.random.randint(0, self.size, batch)
        return (torch.FloatTensor(self.obs[idx]).to(DEVICE),
                torch.FloatTensor(self.act[idx]).to(DEVICE),
                torch.FloatTensor(self.rew[idx]).to(DEVICE),
                torch.FloatTensor(self.obs2[idx]).to(DEVICE),
                torch.FloatTensor(self.done[idx]).to(DEVICE))


print(f"✓ SAC networks defined  (obs_dim={OBS_DIM}, act_dim={ACT_DIM}, hidden={HIDDEN})")
_a = GaussianActor().to(DEVICE)
_c = TwinCritic().to(DEVICE)
_obs_t = torch.zeros(1,OBS_DIM).to(DEVICE)
_act_t = torch.zeros(1,ACT_DIM).to(DEVICE)
print(f"  Actor params : {sum(p.numel() for p in _a.parameters()):,}")
print(f"  Critic params: {sum(p.numel() for p in _c.parameters()):,}")
del _a,_c,_obs_t,_act_t


✓ SAC networks defined  (obs_dim=39, act_dim=4, hidden=(400, 300))
  Actor params : 138,708
  Critic params: 276,402


In [12]:
# ═══════════════════════════════════════════════════════════════════════
#  SAC Agent — paper hyperparameters (FNPG-NH §IV)
# ═══════════════════════════════════════════════════════════════════════

class SACAgent:
    def __init__(self, obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden=HIDDEN,
                 lr=3e-4, gamma=0.99, tau=0.005,
                 buffer_size=1_000_000, batch_size=256,
                 target_entropy=None):
        self.gamma  = gamma
        self.tau    = tau
        self.batch  = batch_size
        self.updates = 0

        self.actor    = GaussianActor(obs_dim,act_dim,hidden).to(DEVICE)
        self.critic   = TwinCritic(obs_dim,act_dim,hidden).to(DEVICE)
        self.critic_t = TwinCritic(obs_dim,act_dim,hidden).to(DEVICE)
        self.critic_t.load_state_dict(self.critic.state_dict())
        for p in self.critic_t.parameters(): p.requires_grad_(False)

        self.opt_a = optim.Adam(self.actor.parameters(),  lr=lr)
        self.opt_c = optim.Adam(self.critic.parameters(), lr=lr)

        # Auto-tune entropy coefficient
        self.log_alpha = torch.zeros(1, requires_grad=True, device=DEVICE)
        self.target_ent = target_entropy if target_entropy else -float(act_dim)
        self.opt_e = optim.Adam([self.log_alpha], lr=lr)

        self.buffer = ReplayBuffer(buffer_size, obs_dim, act_dim)

    @property
    def alpha(self): return self.log_alpha.exp().item()

    def select_action(self, obs, deterministic=False):
        with torch.no_grad():
            o = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
            a, _ = self.actor(o, deterministic=deterministic)
        return a.squeeze(0).cpu().numpy()

    def update(self):
        if self.buffer.size < self.batch:
            return None
        obs, act, rew, obs2, done = self.buffer.sample(self.batch)

        # ── Critic update ─────────────────────────────────────────────
        with torch.no_grad():
            a2, lp2 = self.actor(obs2)
            q1t,q2t = self.critic_t(obs2, a2)
            q_tgt   = torch.min(q1t,q2t) - self.alpha*lp2
            y       = rew + self.gamma*(1-done)*q_tgt

        q1,q2 = self.critic(obs,act)
        loss_c = F.mse_loss(q1,y) + F.mse_loss(q2,y)
        self.opt_c.zero_grad(); loss_c.backward(); self.opt_c.step()

        # ── Actor update ──────────────────────────────────────────────
        a_new, lp_new = self.actor(obs)
        q_pi = self.critic.q_min(obs, a_new)
        loss_a = (self.alpha*lp_new - q_pi).mean()
        self.opt_a.zero_grad(); loss_a.backward(); self.opt_a.step()

        # ── Entropy coeff update ──────────────────────────────────────
        loss_e = -(self.log_alpha*(lp_new + self.target_ent).detach()).mean()
        self.opt_e.zero_grad(); loss_e.backward(); self.opt_e.step()

        # ── Soft target update ─────────────────────────────────────────
        for p,pt in zip(self.critic.parameters(), self.critic_t.parameters()):
            pt.data.mul_(1-self.tau).add_(self.tau*p.data)

        self.updates += 1
        return dict(loss_c=loss_c.item(), loss_a=loss_a.item(), alpha=self.alpha)

print("✓ SACAgent defined")


✓ SACAgent defined


In [13]:
# ═══════════════════════════════════════════════════════════════════════
#  Training configuration & main loop
# ═══════════════════════════════════════════════════════════════════════

CFG = dict(
    total_steps    = 50_000,   # increase to 1_000_000 for best results
    start_steps    = 2_000,     # random exploration before learning
    update_every   = 1,
    updates_per_step = 1,
    batch_size     = 256,
    buffer_size    = 1_000_000,
    lr             = 3e-4,
    gamma          = 0.99,
    tau            = 0.005,
    log_every      = 5_000,
    eval_every     = 25_000,
    eval_episodes  = 20,
    dt             = 0.05,
    max_t          = 50.0,
    w_agg          = 5.0,    # primary aggressiveness weight
    w_ocp          = 2.0,    # OCP imitation weight
    w_ctrl         = 0.05,   # control smoothness weight
    ocp_seed_prob  = 0.70,   # fraction of episodes seeded from OCP library
)

MODEL_FILE   = 'f18_sac_aggressive.pt'
FORCE_RETRAIN = False   # set True to re-train from scratch

def aggressiveness_index(states):
    """AI = mean |p|+|q|+|r| [deg/s] + RMS rate-of-change."""
    pqr = np.degrees(states[:, 3:6])
    return np.mean(np.abs(pqr).sum(axis=1)) + np.sqrt(np.mean(np.diff(pqr,axis=0)**2))

def evaluate_agent(agent, env, n_ep=20):
    returns = []; precision = []; tight = []; loose = []; scores = []

    for _ in range(n_ep):
        obs, _ = env.reset()
        done = False; ep_r = 0.
        ep_prec = ep_tight = ep_loose = False

        while not done:
            a = agent.select_action(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(a)
            ep_r += r; done = term or trunc
            ep_prec  |= info.get('in_precision', False)
            ep_tight |= info.get('in_tight',     False)
            ep_loose |= info.get('in_loose',     False)

        returns.append(ep_r)
        precision.append(float(ep_prec))
        tight.append(float(ep_tight))
        loose.append(float(ep_loose))
        # weighted success score
        sc = 1.0 if ep_prec else (0.8 if ep_tight else (0.4 if ep_loose else 0.0))
        scores.append(sc)

    print(f"  Return   : {np.mean(returns):.1f} ± {np.std(returns):.1f}")
    print(f"  Precision: {np.mean(precision):.2f}  "
          f"Tight: {np.mean(tight):.2f}  "
          f"Loose: {np.mean(loose):.2f}")
    print(f"  Weighted success score: {np.mean(scores):.3f}")
    return np.mean(returns), np.mean(scores), np.mean(precision)

def train_sac(cfg, env):
    agent = SACAgent(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden=HIDDEN,
                     lr=cfg['lr'], gamma=cfg['gamma'], tau=cfg['tau'],
                     buffer_size=cfg['buffer_size'], batch_size=cfg['batch_size'])

    print("=" * 65)
    print("SAC Training — Aggressiveness-only (OCP-as-reward, NO BC)")
    print("=" * 65)
    print(f"  total_steps={cfg['total_steps']:,}  start_steps={cfg['start_steps']:,}")
    print(f"  w_agg={cfg['w_agg']}  w_ocp={cfg['w_ocp']}  ocp_seed_prob={cfg['ocp_seed_prob']}")
    print()

    obs,_ = env.reset(); ep_r=0.; ep_steps=0; ep_num=0
    metrics = dict(returns=[],succs=[],loss_c=[],loss_a=[],alphas=[],ai=[])
    eval_log = []
    t0 = time.time()

    for step in range(1, cfg['total_steps']+1):

        # Collect experience
        if step < cfg['start_steps']:
            act = env.action_space.sample()
        else:
            act = agent.select_action(obs, deterministic=False)

        obs2, rew, term, trunc, info = env.step(act)
        done = term or trunc
        agent.buffer.push(obs, act, rew, obs2, float(term))
        obs = obs2; ep_r += rew; ep_steps += 1

        if done:
            metrics['returns'].append(ep_r)
            metrics['succs'].append(float(term))
            obs,_ = env.reset(); ep_r=0.; ep_steps=0; ep_num+=1

        # Update
        if step >= cfg['start_steps'] and step % cfg['update_every'] == 0:
            for _ in range(cfg['updates_per_step']):
                iu = agent.update()
                if iu:
                    metrics['loss_c'].append(iu['loss_c'])
                    metrics['loss_a'].append(iu['loss_a'])
                    metrics['alphas'].append(iu['alpha'])

        # Logging
        if step % cfg['log_every'] == 0:
            w = min(50, len(metrics['returns']))
            mr = np.mean(metrics['returns'][-w:]) if metrics['returns'] else float('nan')
            ms = np.mean(metrics['succs'][-w:])   if metrics['succs']   else float('nan')
            mc = np.mean(metrics['loss_c'][-100:]) if metrics['loss_c'] else float('nan')
            al = np.mean(metrics['alphas'][-100:]) if metrics['alphas'] else float('nan')
            print(f"  Step {step:>7,} | Ep{ep_num:>5,} | "
                  f"RetMean={mr:>8.1f} | Succ={ms:.2f} | "
                  f"Lc={mc:.4f} | α={al:.3f} | {(time.time()-t0)/60:.1f}m")

        # Evaluate
        if step % cfg['eval_every'] == 0:
            mr,sr,sc,ai = evaluate_agent(agent, env, cfg['eval_episodes'])
            eval_log.append((step,mr,sr,sc,ai))
            print(f"  ► EVAL step={step:,}  return={mr:.1f}±{sr:.1f}  "
                  f"success={sc:.2f}  AI={ai:.1f}°/s")
            torch.save({'actor':agent.actor.state_dict(),
                        'critic':agent.critic.state_dict(),
                        'log_alpha':agent.log_alpha.item(),
                        'step':step}, MODEL_FILE)

    return agent, metrics, eval_log


# ── Instantiate ──
ENV = F18AggressiveEnv(
    TRIM_LIBRARY, TRANSITION_LIBRARY, FEASIBLE_PAIRS,
    dt=CFG['dt'], max_t=CFG['max_t'],
    ocp_seed_prob=CFG['ocp_seed_prob'],
    w_agg=CFG['w_agg'], w_ocp=CFG['w_ocp'], w_ctrl=CFG['w_ctrl'])

if os.path.exists(MODEL_FILE) and not FORCE_RETRAIN:
    ckpt = torch.load(MODEL_FILE, map_location=DEVICE)
    AGENT = SACAgent(obs_dim=OBS_DIM, act_dim=ACT_DIM, hidden=HIDDEN)
    AGENT.actor.load_state_dict(ckpt['actor'])
    AGENT.critic.load_state_dict(ckpt['critic'])
    AGENT.log_alpha.data.fill_(ckpt.get('log_alpha', 0.0))
    print(f"✓ Loaded checkpoint '{MODEL_FILE}'  (step={ckpt.get('step','?')})")
    METRICS, EVAL_LOG = {}, []
else:
    print(f"Training from scratch — {CFG['total_steps']:,} steps")
    AGENT, METRICS, EVAL_LOG = train_sac(CFG, ENV)
    print("\n✓ Training complete")


Training from scratch — 50,000 steps
SAC Training — Aggressiveness-only (OCP-as-reward, NO BC)
  total_steps=50,000  start_steps=2,000
  w_agg=5.0  w_ocp=2.0  ocp_seed_prob=0.7

  Step   5,000 | Ep    5 | RetMean=  4458.4 | Succ=0.00 | Lc=0.4093 | α=0.416 | 2.1m
  Step  10,000 | Ep   10 | RetMean=  4527.3 | Succ=0.00 | Lc=0.4001 | α=0.096 | 5.7m
  Step  15,000 | Ep   15 | RetMean=  4542.6 | Succ=0.00 | Lc=0.4574 | α=0.026 | 9.3m
  Step  20,000 | Ep   20 | RetMean=  4552.2 | Succ=0.00 | Lc=0.4759 | α=0.012 | 12.8m
  Step  25,000 | Ep   25 | RetMean=  4565.3 | Succ=0.00 | Lc=0.4599 | α=0.011 | 16.3m
  Return   : 5163.3 ± 1616.1
  Precision: 0.05  Tight: 0.10  Loose: 0.10
  Weighted success score: 0.090


ValueError: not enough values to unpack (expected 4, got 3)

In [14]:
# ═══════════════════════════════════════════════════════════════════════
#  Training diagnostics
# ═══════════════════════════════════════════════════════════════════════
def plot_training(metrics, eval_log):
    if not metrics.get('returns'):
        print("No training metrics to plot (loaded from checkpoint).")
        return
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle('SAC Training — F-18 Aggressive Trajectory (OCP-as-reward)',
                 fontsize=13, fontweight='bold')

    def smooth(arr, w=50):
        w = min(w, len(arr))
        return np.convolve(arr, np.ones(w)/w, mode='valid'), w-1

    ax = axes[0,0]
    ep_rets = metrics['returns']
    sm, off = smooth(ep_rets)
    ax.plot(ep_rets, alpha=0.15, color='steelblue')
    ax.plot(range(off, len(ep_rets)), sm, 'b-', lw=2)
    ax.set_title('Episode Returns'); ax.set_xlabel('Episode'); ax.grid(True,alpha=0.3)

    ax = axes[0,1]
    sm, off = smooth(metrics['succs'])
    ax.plot(range(off,len(metrics['succs'])), sm, 'g-', lw=2)
    ax.set_title('Success Rate (rolling 50)'); ax.set_ylim(0,1); ax.grid(True,alpha=0.3)

    ax = axes[0,2]
    if metrics.get('loss_c'):
        sm, off = smooth(metrics['loss_c'],100)
        ax.plot(range(off,len(metrics['loss_c'])), sm, 'r-', lw=2)
    ax.set_title('Critic Loss'); ax.grid(True,alpha=0.3)

    ax = axes[1,0]
    if metrics.get('loss_a'):
        sm, off = smooth(metrics['loss_a'],100)
        ax.plot(range(off,len(metrics['loss_a'])), sm, 'm-', lw=2)
    ax.set_title('Actor Loss'); ax.grid(True,alpha=0.3)

    ax = axes[1,1]
    if metrics.get('alphas'):
        ax.plot(metrics['alphas'], 'k-', alpha=0.4, lw=1)
    ax.set_title('Entropy α'); ax.grid(True,alpha=0.3)

    ax = axes[1,2]
    if eval_log:
        steps = [e[0] for e in eval_log]
        ais   = [e[4] for e in eval_log]
        succs = [e[3] for e in eval_log]
        ax.plot(steps, ais, 'b-o', lw=2, label='AI (°/s)')
        ax2 = ax.twinx()
        ax2.plot(steps, succs, 'g--s', lw=2, label='Success')
        ax2.set_ylim(0,1)
        ax.set_title('Eval: Aggressiveness & Success')
        ax.set_xlabel('Step'); ax.legend(loc='lower left')
        ax2.legend(loc='lower right')
    ax.grid(True,alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.close()
    print("✓ Saved training_curves.png")

plot_training(METRICS, EVAL_LOG)


NameError: name 'METRICS' is not defined

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  OCP vs RL trajectory comparison
# ═══════════════════════════════════════════════════════════════════════

def rollout_policy(agent, env, i_src, j_tgt, deterministic=True):
    """Roll out for a specific (i,j) pair. Returns states, controls, rewards."""
    obs, _ = env.reset()
    # Override to this specific pair
    env.i_src = i_src; env.j_tgt = j_tgt
    env.x      = env.trim_lib[i_src]['trim_state'].copy()
    env.x_goal = env.trim_lib[j_tgt]['trim_state'].copy()
    env.u_prev = env.trim_lib[i_src]['trim_controls'].copy()
    env.x[3:6] = 0.0
    env.x[6:10] /= max(np.linalg.norm(env.x[6:10]), 1e-12)
    env._prev_dist = np.linalg.norm(env.x[10:13]-env.x_goal[10:13])
    ocp = env.trans_lib.get((i_src,j_tgt),{})
    env._ocp = ocp if ocp.get('feasible') else None
    env.step_n = 0
    obs = env._get_obs()

    states=[env.x.copy()]; ctrls=[]; rews=[]; done=False
    while not done:
        a = agent.select_action(obs, deterministic=deterministic)
        obs,r,term,trunc,info = env.step(a)
        states.append(env.x.copy()); ctrls.append(env.u_prev.copy()); rews.append(r)
        done = term or trunc
    return np.array(states), np.array(ctrls), np.array(rews)


def compare_ocp_rl(agent, env, i, j, save=True):
    """Side-by-side OCP vs RL comparison plot."""
    ref     = env.trans_lib[(i,j)]
    rl_s, rl_u, rl_r = rollout_policy(agent, env, i, j)
    lbl_i   = env.trim_lib[i]['label']
    lbl_j   = env.trim_lib[j]['label']
    t_rl    = np.arange(len(rl_s)) * env.dt

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    fig.suptitle(f'OCP vs RL — {lbl_i} → {lbl_j}', fontsize=13, fontweight='bold')
    C = {'OCP':'#1f77b4', 'RL':'#d62728'}

    # 3-D path
    ax = fig.add_subplot(2, 3, 1, projection='3d')
    if ref['feasible'] and ref['X'] is not None:
        Xr = ref['X']
        ax.plot(Xr[:,10],Xr[:,11],-Xr[:,12], color=C['OCP'], lw=2, label='OCP')
    ax.plot(rl_s[:,10],rl_s[:,11],-rl_s[:,12], color=C['RL'], lw=2, ls='--', label='RL')
    ax.scatter(*env.trim_lib[i]['trim_state'][10:13]*[1,1,-1], c='g', s=80)
    ax.scatter(*env.trim_lib[j]['trim_state'][10:13]*[1,1,-1], c='r', s=80)
    ax.set_title('3-D Path'); ax.legend(fontsize=8)

    # Body rates
    ax = axes[0,1]
    for k,(col,lbl_r) in enumerate([('r','p'),('g','q'),('b','r')]):
        if ref['feasible'] and ref['X'] is not None:
            ax.plot(ref['T'], np.degrees(ref['X'][:,3+k]),
                    color=col, lw=2, alpha=0.4, label=f'OCP-{lbl_r}')
        ax.plot(t_rl, np.degrees(rl_s[:,3+k]),
                color=col, lw=2, ls='--', label=f'RL-{lbl_r}')
    ax.set_title('Body Rates [°/s]'); ax.set_xlabel('t [s]'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

    # Airspeed
    ax = axes[0,2]
    if ref['feasible'] and ref['X'] is not None:
        ax.plot(ref['T'], np.sqrt(np.sum(ref['X'][:,:3]**2,1)), color=C['OCP'], lw=2, label='OCP')
    ax.plot(t_rl, np.sqrt(np.sum(rl_s[:,:3]**2,1)), color=C['RL'], lw=2, ls='--', label='RL')
    ax.set_title('Airspeed [m/s]'); ax.set_xlabel('t [s]'); ax.legend(); ax.grid(True,alpha=0.3)

    # Thrust
    ax = axes[1,0]
    t_u = np.arange(len(rl_u))*env.dt
    if ref['feasible'] and ref['U'] is not None:
        ax.plot(ref['T'][:len(ref['U'])], ref['U'][:,0]/1e3, color=C['OCP'], lw=2, label='OCP')
    ax.plot(t_u, rl_u[:,0]/1e3, color=C['RL'], lw=2, ls='--', label='RL')
    ax.set_title('Thrust [kN]'); ax.set_xlabel('t [s]'); ax.legend(); ax.grid(True,alpha=0.3)

    # Control surfaces
    ax = axes[1,1]
    for k,(col,nm) in enumerate([('r','δe'),('g','δa'),('b','δr')]):
        if ref['feasible'] and ref['U'] is not None:
            ax.plot(ref['T'][:len(ref['U'])], np.degrees(ref['U'][:,k+1]),
                    color=col, lw=2, alpha=0.4, label=f'OCP-{nm}')
        ax.plot(t_u, np.degrees(rl_u[:,k+1]),
                color=col, lw=2, ls='--', label=f'RL-{nm}')
    ax.set_title('Control Surfaces [°]'); ax.set_xlabel('t [s]'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)

    # Step rewards
    ax = axes[1,2]
    ax.plot(t_u, rl_r, color='purple', lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.fill_between(t_u, rl_r, 0, where=np.array(rl_r)>0, alpha=0.3, color='g')
    ax.fill_between(t_u, rl_r, 0, where=np.array(rl_r)<0, alpha=0.3, color='r')
    ax.set_title('Step Rewards'); ax.set_xlabel('t [s]'); ax.grid(True,alpha=0.3)

    # Aggressiveness stats
    ai_rl  = aggressiveness_index(rl_s)
    ai_ocp = aggressiveness_index(ref['X']) if ref['feasible'] and ref['X'] is not None else np.nan
    fig.text(0.5, 0.01,
             f"RL total return={sum(rl_r):.1f}  |  AI(RL)={ai_rl:.1f}°/s  |  "
             f"AI(OCP)={ai_ocp:.1f}°/s  |  ΔAI={ai_rl-ai_ocp:+.1f}°/s",
             ha='center', fontsize=10)

    plt.tight_layout(rect=[0,0.03,1,1])
    fname = f'compare_{lbl_i}_{lbl_j}.png'
    if save: plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Saved {fname}  |  AI(RL)={ai_rl:.1f}  AI(OCP)={ai_ocp:.1f}°/s")


print("Running OCP vs RL comparisons for first 5 feasible pairs ...")
count = 0
for (i,j) in FEASIBLE_PAIRS:
    if TRANSITION_LIBRARY[(i,j)]['feasible']:
        compare_ocp_rl(AGENT, ENV, i, j)
        count += 1
        if count >= 5: break
print(f"✓ {count} comparison plots generated")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  Generalisation evaluation across a subset of feasible pairs
# ═══════════════════════════════════════════════════════════════════════

def generalisation_eval(agent, env, pairs, n_repeat=3, max_pairs=200):
    pairs_eval = pairs[:max_pairs]
    results = []
    print(f"{'Pair':<18} {'Succ':>6} {'Return':>9} {'AI-RL':>8} {'AI-OCP':>8} {'ΔAI%':>8}")
    print("─"*65)
    for (i,j) in pairs_eval:
        pi = env.trim_lib[i]['label']; pj = env.trim_lib[j]['label']
        ep_r=[]; ep_s=[]; ai_rl=[]
        for rep in range(n_repeat):
            random.seed(rep)
            st,_,rws = rollout_policy(agent, env, i, j, deterministic=True)
            ep_r.append(sum(rws)); ep_s.append(float(env._prev_dist < 60.))
            ai_rl.append(aggressiveness_index(st))
        ai_rl_m = np.mean(ai_rl)
        ref = env.trans_lib[(i,j)]
        ai_ocp = aggressiveness_index(ref['X']) if ref['feasible'] and ref['X'] is not None else np.nan
        d_pct  = (ai_rl_m-ai_ocp)/max(abs(ai_ocp),1)*100 if np.isfinite(ai_ocp) else float('nan')
        row = dict(i=i,j=j,pair=f'{pi}→{pj}',success=np.mean(ep_s),
                   ret_mean=np.mean(ep_r), ai_rl=ai_rl_m, ai_ocp=ai_ocp, delta_ai=d_pct)
        results.append(row)
        print(f"  {row['pair']:<16} {row['success']:>6.2f} {row['ret_mean']:>9.1f} "
              f"{row['ai_rl']:>8.1f} "
              f"{'N/A':>8}" if not np.isfinite(ai_ocp) else
              f"  {row['pair']:<16} {row['success']:>6.2f} {row['ret_mean']:>9.1f} "
              f"{row['ai_rl']:>8.1f} {row['ai_ocp']:>8.1f} {row['delta_ai']:>+8.1f}%")

    # Summary
    print()
    n_ok  = sum(r['success']>=0.67 for r in results)
    m_ret = np.mean([r['ret_mean'] for r in results])
    m_ai  = np.mean([r['ai_rl'] for r in results])
    feas  = [r for r in results if np.isfinite(r['delta_ai'])]
    m_dai = np.nanmean([r['delta_ai'] for r in feas]) if feas else float('nan')
    print("═"*65)
    print(f"  Pairs evaluated     : {len(results)}")
    print(f"  Success (≥67%)      : {n_ok} ({100*n_ok/max(len(results),1):.1f}%)")
    print(f"  Mean episode return : {m_ret:.1f}")
    print(f"  Mean AI (RL)        : {m_ai:.1f} °/s")
    print(f"  Mean ΔAI vs OCP     : {m_dai:+.1f}%  (+ = more aggressive than OCP)")
    print("═"*65)
    return results


print("Generalisation evaluation (first 200 feasible pairs) ...")
GEN_RESULTS = generalisation_eval(AGENT, ENV, FEASIBLE_PAIRS[:200])


In [ ]:
# ── Generalisation heatmap ──────────────────────────────────────────────
def plot_heatmap(results, trim_lib, metric='success'):
    N = len(trim_lib); labels = [p['label'] for p in trim_lib]
    mat = np.full((N,N), np.nan)
    for r in results:
        mat[r['i'], r['j']] = r[metric]
    fig, ax = plt.subplots(figsize=(12, 10))
    cm = 'RdYlGn' if metric=='success' else 'plasma'
    im = ax.imshow(mat, cmap=cm, aspect='auto',
                   vmin=0 if metric=='success' else None,
                   vmax=1 if metric=='success' else None)
    ax.set_xticks(range(N)); ax.set_xticklabels(labels, rotation=90, fontsize=5)
    ax.set_yticks(range(N)); ax.set_yticklabels(labels, fontsize=5)
    title_map = {'success':'Success Rate','ai_rl':'Aggressiveness Index (RL)','ret_mean':'Mean Return'}
    ax.set_title(f'Generalisation — {title_map.get(metric,metric)}', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    fname = f'heatmap_{metric}.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight'); plt.close()
    print(f"  Saved {fname}")

for m in ['success','ai_rl','ret_mean']:
    plot_heatmap(GEN_RESULTS, TRIM_LIBRARY, m)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  Fast inference — any (x0, xf) in ~0.04 s
#  No re-training, no OCP solve.
# ═══════════════════════════════════════════════════════════════════════

def get_aggressive_trajectory(agent, x0, xf,
                               dt=0.05, max_steps=600,
                               goal_vel_tol=8.0, goal_pos_tol=60.0):
    """
    Generate an aggressive trajectory from x0 to xf using the trained SAC policy.

    Parameters
    ----------
    agent      : trained SACAgent
    x0, xf    : np.array(13,) — start and goal trim states
    dt         : simulation timestep [s]
    max_steps  : maximum number of steps before timeout

    Returns
    -------
    X_traj : (N+1, 13)  state trajectory
    U_traj : (N,   4)   control sequence
    T_traj : (N+1,)     time axis [s]
    info   : dict with pqr_rms, duration, goal_reached
    """
    X_SCALE = F18AggressiveEnv.X_SCALE
    U_MID   = F18AggressiveEnv.U_MID
    U_HALF  = F18AggressiveEnv.U_HALF

    x = x0.copy()
    x[3:6]  = 0.0
    x[6:10] /= max(np.linalg.norm(x[6:10]), 1e-12)

    states=[x.copy()]; controls=[]

    t_start = time.perf_counter()
    for _ in range(max_steps):
        obs = np.concatenate([
            x / X_SCALE, xf / X_SCALE, (x-xf) / X_SCALE
        ], dtype=np.float32)

        action, _ = agent.actor(
            torch.FloatTensor(obs).unsqueeze(0).to(DEVICE),
            deterministic=True)
        action = action.squeeze(0).detach().cpu().numpy()

        u = np.clip(
            U_MID + action*U_HALF,
            [A.thrust_min, A.delta_e_min, A.delta_a_min, A.delta_r_min],
            [A.thrust_max, A.delta_e_max, A.delta_a_max, A.delta_r_max])

        x = rk4_step(x, u, dt)
        states.append(x.copy()); controls.append(u.copy())

        vel_err = np.linalg.norm(x[:6]-xf[:6])
        pos_err = np.linalg.norm(x[10:13]-xf[10:13])
        if vel_err < goal_vel_tol and pos_err < goal_pos_tol:
            break

    t_inf = (time.perf_counter()-t_start)*1000  # ms

    X_traj = np.array(states)
    U_traj = np.array(controls)
    T_traj = np.arange(len(states))*dt

    pqr_rms = np.degrees(np.sqrt(np.mean(X_traj[:,3:6]**2)))
    ai      = aggressiveness_index(X_traj)
    info = dict(pqr_rms_deg=pqr_rms, ai=ai, duration_s=T_traj[-1],
                goal_reached=(vel_err<goal_vel_tol and pos_err<goal_pos_tol),
                inference_ms=t_inf, n_steps=len(controls))
    return X_traj, U_traj, T_traj, info


# ── Demo: unseen pair ─────────────────────────────────────────────────
print("=== Fast Inference Demo ===")
print()

# Seeded pairs from library
for ii, jj in [(0,50),(50,100),(100,200)]:
    x0 = TRIM_LIBRARY[ii]['trim_state']
    xf = TRIM_LIBRARY[jj]['trim_state']
    X_t, U_t, T_t, nfo = get_aggressive_trajectory(AGENT, x0, xf)
    print(f"  Pair ({ii:>3},{jj:>3})  |  "
          f"steps={nfo['n_steps']:>4}  |  dur={nfo['duration_s']:.1f}s  |  "
          f"AI={nfo['ai']:.1f}°/s  |  "
          f"reached={nfo['goal_reached']}  |  "
          f"inference={nfo['inference_ms']:.1f}ms")

print()
print("─" * 65)

# True unseen pair — interpolated start state (not in any trim state)
x0_new = TRIM_LIBRARY[10]['trim_state'].copy()
x0_new[:3] *= 1.05   # 5% velocity perturbation
x0_new[6:10] /= np.linalg.norm(x0_new[6:10])
xf_new = TRIM_LIBRARY[150]['trim_state']

X_new, U_new, T_new, nfo_new = get_aggressive_trajectory(AGENT, x0_new, xf_new)
print(f"\nUnseen perturbed pair:")
print(f"  steps={nfo_new['n_steps']}  dur={nfo_new['duration_s']:.1f}s  "
      f"AI={nfo_new['ai']:.1f}°/s  reached={nfo_new['goal_reached']}  "
      f"inference={nfo_new['inference_ms']:.2f}ms")
print()
print(f"✓ RL inference ~{nfo_new['inference_ms']:.1f}ms  vs OCP ~5-25 min")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  Save & Load model
# ═══════════════════════════════════════════════════════════════════════

def save_model(agent, path='f18_sac_aggressive_final.pt'):
    torch.save({
        'actor'    : agent.actor.state_dict(),
        'critic'   : agent.critic.state_dict(),
        'log_alpha': agent.log_alpha.item(),
        'updates'  : agent.updates,
        'obs_dim'  : OBS_DIM,
        'act_dim'  : ACT_DIM,
        'hidden'   : HIDDEN,
    }, path)
    print(f"✓ Model saved → '{path}'  (updates={agent.updates:,})")


def load_model(path='f18_sac_aggressive_final.pt'):
    ckpt = torch.load(path, map_location=DEVICE)
    agent = SACAgent(
        obs_dim=ckpt.get('obs_dim', OBS_DIM),
        act_dim=ckpt.get('act_dim', ACT_DIM),
        hidden =ckpt.get('hidden',  HIDDEN))
    agent.actor.load_state_dict(ckpt['actor'])
    agent.critic.load_state_dict(ckpt['critic'])
    agent.log_alpha.data.fill_(ckpt.get('log_alpha', 0.0))
    agent.updates = ckpt.get('updates', 0)
    print(f"✓ Model loaded from '{path}'  (updates={agent.updates:,})")
    return agent


save_model(AGENT)

print()
print("─── Quick usage ─────────────────────────────────────────────────")
print("  agent = load_model('f18_sac_aggressive_final.pt')")
print("  X, U, T, info = get_aggressive_trajectory(agent, x0, xf)")
print("  # inference in ~0.04 s, generalises to all 275×275 pairs")
